In [1]:
# Setup: repo path and single-paper config (with optional env override).
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

PAPER_FILENAME = os.getenv("PAPER_FILENAME", "2.md")
DOMAIN_NAMES = [f"domain_{i}" for i in range(1, 8)]

print(f"Using paper: {PAPER_FILENAME}")
print(f"Domains: {', '.join(DOMAIN_NAMES)}")

Using paper: 2.md
Domains: domain_1, domain_2, domain_3, domain_4, domain_5, domain_6, domain_7


In [2]:
# Imports: two prompt builders + LLM call helper.
import json

from domain_shot import build_question_scoring_prompt as qs
from domain_shot import build_tree_guided_prompt as tq
from domain_shot.evaluation import DEFAULT_MODEL, call_llm_and_process

In [3]:
# Load input texts for one paper and all domains.
paper_path = Path(tq.DEFAULT_PAPERS_DIR) / PAPER_FILENAME
paper_text = tq.read_text_file(str(paper_path))
intro_text = tq.load_default_guidance_intro()

domain_questions_by_domain = {}
decision_trees_by_domain = {}
for domain_name in DOMAIN_NAMES:
    domain_questions_by_domain[domain_name] = tq.load_domain_questions(domain_name)
    decision_trees_by_domain[domain_name] = tq.load_domain_decision_tree(domain_name)

print(f"Paper: {paper_path.name}")
print(f"Loaded domains: {len(DOMAIN_NAMES)}")

Paper: 2.md
Loaded domains: 7


In [4]:
# Build both prompt message payloads for all domains.
tree_messages_by_domain = {}
scoring_messages_by_domain = {}

for domain_name in DOMAIN_NAMES:
    tree_messages_by_domain[domain_name] = tq.build_prompt_messages(
        domain_name=domain_name,
        intro_text=intro_text,
        decision_tree_text=decision_trees_by_domain[domain_name],
        domain_questions_text=domain_questions_by_domain[domain_name],
        paper_text=paper_text,
    )
    scoring_messages_by_domain[domain_name] = qs.build_prompt_messages(
        domain_name=domain_name,
        intro_text=intro_text,
        domain_questions_text=domain_questions_by_domain[domain_name],
        paper_text=paper_text,
    )

print("Built tree-guided and question-scoring messages for all domains.")

Built tree-guided and question-scoring messages for all domains.


In [5]:
# Run both prompts on the same paper across all domains.
tree_results = {}
scoring_results = {}

for domain_name in DOMAIN_NAMES:
    tree_results[domain_name] = call_llm_and_process(
        domain_name,
        tree_messages_by_domain[domain_name],
        DEFAULT_MODEL,
    )
    scoring_results[domain_name] = call_llm_and_process(
        domain_name,
        scoring_messages_by_domain[domain_name],
        DEFAULT_MODEL,
    )

print(f"Tree-guided results: {len(tree_results)} domains")
print(f"Question-scoring results: {len(scoring_results)} domains")

Tree-guided results: 7 domains
Question-scoring results: 7 domains


In [6]:
# Save one combined output JSON.
output_dir = repo_root / "output" / "single_paper_all_domains_two_prompts"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{paper_path.stem}.json"

combined_results = {
    "paper": PAPER_FILENAME,
    "domains": DOMAIN_NAMES,
    "tree_guided": tree_results,
    "question_scoring": scoring_results,
}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(combined_results, f, indent=2, ensure_ascii=False)

print(f"Saved: {output_path}")

Saved: /home/user/Thesis/conservation-llm-critical-appraisal/output/single_paper_all_domains_two_prompts/2.json
